# Pytorch tutorials

## 6 - Optimization loop

### First steps

Libs

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

Manage data

In [2]:
training_data = datasets.FashionMNIST( # training data
    root="data", # data directory
    train=True, # training data
    download=True, # download if not exist
    transform=ToTensor() # convert the image to tensor
)

test_data = datasets.FashionMNIST( # test data
    root="data",
    train=False,
    download=True, # download if not exist
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64) # load the training data AND fix the batch size
test_dataloader = DataLoader(test_data, batch_size=64) # load the test data  AND fix the batch size


Define class and an instance 

In [3]:
# A class to define the neural network
class NeuralNetwork(nn.Module): # define the neural network
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() # flatten the tensor
        self.linear_relu_stack = nn.Sequential( # define the neural network
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x): # forward pass
        x = self.flatten(x) # flatten the tensor
        logits = self.linear_relu_stack(x) # pass the tensor through the neural network
        return logits # return the output

# An instance of the neural network
model = NeuralNetwork()

### Hyperparameters

In [4]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

### The loss function

Is the tool used to assess the current model. \
In other words, to assess the current parmaters of the model. \
It is a measure of the distance of our predictions and the actual test output values. 

In [5]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

### The optimizer 
Is the method used to descend the loss function in order to get the best paramaters. \
In other words, given a set of parameters, we have a value of the loss function. \
Given this object, we want to find the paramters that minimize the value of the loss function. To do so, we need a numerical method. \
In our case, we use the stochastic gradient descent (SGD) on the `model paramters` with batch size of 64 (specified in the Data loader), with a specified learning rate (`lr`).


In [7]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

### Full implementation 

#### All
Here there is a function for the training and testing parts.

#### Focus 
I want to focus on the following part of the train function:
```python
# Backpropagation
loss.backward() # backpropagate the loss
optimizer.step() # update the weights
optimizer.zero_grad() # zero the gradients
```

A batch is chosen, then we take some data of the size of the batch, we do predictions (using the current paramters of the model) and we compute the loss function (let say the distance from the actual data). \
We save the value of the current loss function in loss. \
Now, we apply `loss.backward` that compute the gradient of the loss function wrt each paramter of the model. \
The value of these gradients is implicetely stored in  `.grad` that is a feature of each paramter of the model.   

`optimizer.step()` updates the value of the paramters of the model using the current gradients. \
The optimizer was initilized with giving to it `model.paramters()`. \
So, it has access to the attribute of each paramter `.grad`. \
For this reason, now the optimizer can update the paramters() depdning on the loss data. 

`optimizer.zero_grad()` puts the values of the grads to zero. \
Each paramter of the model has an attribut grad and the purpose of optimizer.zero_grad is to put all of them equal to zero. \
The aim is to avoid error and be able to call again these 3 functions for the next step of the optimization.


In [9]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset) # get the size of the dataset
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train() # set the model to training mode
    for batch, (X, y) in enumerate(dataloader): # take the first batch of the data 
        # Compute prediction and loss
        pred = model(X) # pass the input through the model
        loss = loss_fn(pred, y) # calculate the loss

        # Backpropagation
        loss.backward() # backpropagate the loss
        optimizer.step() # update the weights
        optimizer.zero_grad() # zero the gradients

        if batch % 100 == 0: # print the loss every 100 batches
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn): # test the model
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval() # set the model to evaluation mode
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad(): # do not calculate the gradients
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### Model usage 

In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.309285  [   64/60000]
loss: 2.293981  [ 6464/60000]
loss: 2.275874  [12864/60000]
loss: 2.265313  [19264/60000]
loss: 2.264357  [25664/60000]
loss: 2.228572  [32064/60000]
loss: 2.235947  [38464/60000]
loss: 2.206997  [44864/60000]
loss: 2.212576  [51264/60000]
loss: 2.185157  [57664/60000]
Test Error: 
 Accuracy: 42.8%, Avg loss: 2.178339 

Epoch 2
-------------------------------
loss: 2.190979  [   64/60000]
loss: 2.181400  [ 6464/60000]
loss: 2.124439  [12864/60000]
loss: 2.135391  [19264/60000]
loss: 2.107724  [25664/60000]
loss: 2.038616  [32064/60000]
loss: 2.069183  [38464/60000]
loss: 1.997435  [44864/60000]
loss: 2.011784  [51264/60000]
loss: 1.942760  [57664/60000]
Test Error: 
 Accuracy: 57.9%, Avg loss: 1.939524 

Epoch 3
-------------------------------
loss: 1.975241  [   64/60000]
loss: 1.945853  [ 6464/60000]
loss: 1.827475  [12864/60000]
loss: 1.860077  [19264/60000]
loss: 1.773839  [25664/60000]
loss: 1.705102  [32064/600